<!-- nav -->
[← 8.01 · Python Types](08_01_Python_Types.ipynb) · [Home](../../README.md) · [8.03 · Python RegExp →](08_03_Python_RegExp.ipynb)

# 8.02 · Python Patterns

Python 3.10's `match` statement compares a value with patterns, one per `case`, and binds names where one matches:

```text
match shape:
    case Point(x=0, y=0):              print("origin")
    case [first, *rest]:               print(first, rest)
    case {"kind": "circle", **extra}:  print(extra)
    case _:                            print("something else")
```

`Point(x=0, y=0)`, `[first, *rest]` and `{"kind": "circle", **extra}` look like expressions, but are not: a pattern is a
grammar of its own (the Language Reference, 8.6.4, "Patterns"), where `[first, *rest]` builds no list but takes one
apart, and `first` reads no name but binds one. `Python Patterns` is its dialect, its eight kinds `ast`'s pattern
nodes, named and shaped as mbse-programs has them
([CAP-276: Python's type expressions and match patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-276.md)).
The `match` statement and its `case`s are statements, mbse-programs'; here, a pattern stands alone, as the root of its
dialect's holder, `S.Python.Patterns.Expression` (tutorial 8, 08.3).

This page builds patterns from a single name up, in the builders' fluent calls, prints each back as Python writes it,
asks what each binds, validates them, and reads the editions that have its kinds. Like the rest of Part I, it starts
from `dialects()`, a store with every dialect installed and defined; `variable` writes a name, as tutorials 5 to 8 do,
and `pattern` holds a pattern's spec in the dialect's holder. A pattern's text is no expression, so no reader here reads
it; where a pattern holds an expression, Python also reads that, with `Python3.Text.FromText` (tutorial 3).

In [1]:
from mbse.Expressions.Framework import Standards, Symbolics, Terms
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3
from toolkit import dialects
from mbse.Expressions.Framework.Simplifiers import NotEvaluated

store = dialects()
S = store.Expressions
variable = lambda name: lambda d: d.variable(lambda v: v.name(name))
PATTERNS = PythonPatterns.Expressions.DIALECT
ToText, binds = PythonPatterns.Text.ToText, PythonPatterns.Expressions.bound_names


def pattern(spec):
    """The pattern `spec` writes, held by its dialect's holder, written in a transaction of its own."""
    with store.transaction():
        return S.Python.Patterns.Expression(lambda e: e.root(spec))


def show(held):
    """A pattern's text, its kind, and the names it binds."""
    term = Terms.term(held)
    print(f"{ToText(held):38}", f"{Terms.kind_of(term)[0]:14}", "binds:", ", ".join(sorted(binds(term))) or "none")

<a id="T08.02.1"></a>
## 08.02.1 · A capture, and the wildcard

The smallest patterns match anything. A **capture** pattern, `first`, matches any value and binds it to its name; the
**wildcard**, `_`, matches any value and binds nothing. Both are one kind, `matchas` (`ast`'s `MatchAs`), with a name
or without; the dialect's `bound_names` gives the names a pattern binds:

In [2]:
capture = lambda name: lambda u: u.matchas(lambda m: m.name(name))  # `name`, a capture pattern
wildcard = lambda u: u.matchas(lambda m: m)  # `_`
for spec in (capture("first"), wildcard):
    show(pattern(spec))
assert [ToText(pattern(spec)) for spec in (capture("first"), wildcard)] == ["first", "_"]

first                                  matchas        binds: first
_                                      matchas        binds: none


<a id="T08.02.2"></a>
## 08.02.2 · Literals, values and singletons

A **value** pattern matches a value equal to its own: a number, a string, or a dotted name such as `Color.RED`, each a
Python3 term the pattern holds as its `value`, a subterm of Python3's union. `None`, `True` and `False` are compared by
identity, so they are **singleton** patterns, their `value` an attribute, one of the three. A dotted name reads a name,
`Color`, which the scope must give; a pattern binds none of them. Python reads each value's text into the same term:

In [3]:
one = lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(1)).text("1").base(10))  # 1, as Python reads it
values = {
    "-1": lambda d: d.unaryop(lambda n: n.op("-").operand(one)),
    "'circle'": lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar("circle")).text("'circle'").quote("'")),
    "Color.RED": lambda d: d.attribute(lambda a: a.value(variable("Color")).attr("RED")),
}
for source, value in values.items():
    held = pattern(lambda u: u.matchvalue(lambda m: m.value(value)))
    show(held)
    with store.transaction():
        read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText(source)))  # Python reads the value
    assert ToText(held) == source and Terms.same(Terms.form(Terms.term(held)).arguments[0], Terms.term(read))
for singleton in ("None", "True"):
    show(pattern(lambda u: u.matchsingleton(lambda m: m.value(singleton))))

-1                                     matchvalue     binds: none
'circle'                               matchvalue     binds: none
Color.RED                              matchvalue     binds: none
None                                   matchsingleton binds: none
True                                   matchsingleton binds: none


<a id="T08.02.3"></a>
## 08.02.3 · Sequences, mappings and classes

Patterns nest, as a value's parts do. A **sequence** pattern, `[first, *rest]`, takes a sequence apart, item by item, a
**star** pattern, `*rest`, binding what is left as a list; its delimiters are `[]`, `()` or none. A **mapping** pattern,
`{"kind": "circle", **extra}`, matches keys, each a literal or a value pattern's name, with a pattern for each, and binds
what is left to its `rest`. A **class** pattern, `Point(0, y=[first, *rest])`, matches an instance of its class, a
Python3 term, and its attributes in order and by keyword, a pattern for each. An **or** pattern, `a | b`, matches
either; an **as** pattern, `[_] as single`, binds the value its pattern matched. Built from those of 08.02.1 and 08.02.2:

In [4]:
star = lambda name: lambda u: u.matchstar(lambda m: m.name(name))  # `*name`
listed = lambda u: u.matchsequence(lambda s: s.delimiters("[]").patterns([capture("first"), star("rest")]))  # [first, *rest]
zero = lambda u: u.matchvalue(lambda m: m.value(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(0)))))  # 0
specs = [
    listed,
    lambda u: u.matchsequence(lambda s: s.delimiters("()").patterns([capture("x"), capture("y")])),
    lambda u: u.matchmapping(lambda m: m.keys([lambda k: k.literal(lambda r: r.value(lambda v: v.as_scalar("kind")))])
        .patterns([lambda p: p.matchvalue(lambda m: m.value(values["'circle'"]))]).rest("extra")),
    lambda u: u.matchclass(lambda c: c.cls(variable("Point")).patterns([zero]).kwd_attrs(["y"]).kwd_patterns([listed])),
    lambda u: u.matchor(lambda o: o.patterns([zero, lambda p: p.matchsingleton(lambda m: m.value("None"))])),
    lambda u: u.matchas(lambda m: m.pattern(lambda p: p.matchsequence(lambda s: s.delimiters("[]").patterns([wildcard])))
        .name("single")),
]
for spec in specs:
    show(pattern(spec))
assert [ToText(pattern(spec)) for spec in specs] == [
    "[first, *rest]", "(x, y)", "{'kind': 'circle', **extra}", "Point(0, y=[first, *rest])", "0 | None", "[_] as single"]

[first, *rest]                         matchsequence  binds: first, rest
(x, y)                                 matchsequence  binds: x, y
{'kind': 'circle', **extra}            matchmapping   binds: extra
Point(0, y=[first, *rest])             matchclass     binds: first, rest
0 | None                               matchor        binds: none
[_] as single                          matchas        binds: single


<a id="T08.02.4"></a>
## 08.02.4 · What a pattern binds, and what it reads

A pattern binds names, but is no binder of its own: none of its kinds refines the framework's `Let` (tutorial 7), since
it is the `case` holding it that binds its names, in its guard and its body. `bound_names` gives the names wherever
they nest: a capture's, a star's, a mapping's rest and an as pattern's. What a pattern reads is its expressions'
names: a class's, a dotted value's, a mapping key's, which are references its scope must give, so `Symbolics.free`
gives those, never what it binds. One pattern of every kind:

In [5]:
every = pattern(lambda u: u.matchor(lambda o: o.patterns([  # Point(0, y=[first, *rest]) | {'k': None, **kw} | (Color.RED as c)
    specs[3],
    lambda p: p.matchmapping(lambda m: m.keys([lambda k: k.literal(lambda r: r.value(lambda v: v.as_scalar("k")))])
        .patterns([lambda p: p.matchsingleton(lambda s: s.value("None"))]).rest("kw")),
    lambda p: p.matchas(lambda m: m.pattern(lambda q: q.matchvalue(lambda v: v.value(values["Color.RED"]))).name("c"))])))
term = Terms.term(every)
print(ToText(every))
print("its kinds:", ", ".join(sorted({Terms.kind_of(t)[0] for t in Terms.walk(term) if Terms.dialect_of(t) == PATTERNS.union})))
print("binds:", ", ".join(sorted(binds(term))), "| reads:", ", ".join(sorted(Symbolics.free(every, PATTERNS))))
print("a binder:", Terms.is_binder(Terms.kind_of(term)[1].as_schema()))
assert binds(term) == {"first", "rest", "kw", "c"} and Symbolics.free(every, PATTERNS) == {"Point", "Color"}

Point(0, y=[first, *rest]) | {'k': None, **kw} | (Color.RED as c)
its kinds: matchas, matchclass, matchmapping, matchor, matchsequence, matchsingleton, matchstar, matchvalue
binds: c, first, kw, rest | reads: Color, Point
a binder: False


<a id="T08.02.5"></a>
## 08.02.5 · Validated

Validation checks each kind as Python's grammar and `ast` do, in its own words: a name must be an identifier, an or
pattern needs two alternatives, a mapping a pattern for each key, a class one for each keyword; and `_` captures
nothing, so a capture or a star named `_` is refused, kept as written and printed
([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)). The
names a pattern reads are reported too, unless its scope gives them:

In [6]:
refusals = [
    lambda u: u.matchas(lambda m: m.name("_")),
    capture("1st"),
    lambda u: u.matchor(lambda o: o.patterns([zero])),
]
for spec in refusals:
    held = pattern(spec)
    print(f"{ToText(held):16}", "; ".join(PATTERNS.validate(held)))
print("; ".join(PATTERNS.validate(every)), "|", PATTERNS.validate(every, {"Point", "Color"}) or "valid")
assert PATTERNS.validate(pattern(refusals[0])) == ["a matchas's name cannot be '_'"]

_                a matchas's name cannot be '_'
1st              a matchas's name must be an identifier, got '1st'
0                a MatchOr needs two patterns
argument 0: cls: variable 'Point' is not bound; argument 2: pattern: value: value: variable 'Color' is not bound | valid


<a id="T08.02.6"></a>
## 08.02.6 · Held by its holder alone, and not simplified

Python3 holds no pattern: a `match` is a statement, and no Python3 expression has a subterm of `Python Patterns`'
`Term`, while a pattern's own subterms hold Python3's terms. Python3 reads `Python Patterns`' kinds, and they read its,
as each dialect of the family reads the others' (`reads()`), and Python3's printer prints them, `Python Patterns`'
`ToText` being its own. A pattern is matched by the `match` statement, not simplified: Python3's rules fold none of its kinds: each stays in the residual, named by its dialect (tutorial 8, 08.3):

In [7]:
python3 = store.registered(Python3.Expressions.DIALECT.union)
held_by = [f"{b.name}.{name}" for b in python3.branches for name, p in b.schema.properties.items()
           if store.registered(PATTERNS.union) in (p.schema, getattr(p.schema, "element", None))]
print("Python3's properties that hold a pattern:", ", ".join(held_by) or "none")
print("Python3 reads", ", ".join(d.name() for d in Python3.Expressions.DIALECT.reads()), "| one printer:", ToText is Python3.Text.ToText)
for held in (pattern(listed), every):
    try:
        Python3.Simplifiers.OfAny(held).evaluate()
        raise AssertionError("evaluated")
    except NotEvaluated as error:
        print(error)

Python3's properties that hold a pattern: none
Python3 reads Python Types, Python Patterns, Python RegExp | one printer: True
Python Patterns matchsequence is not evaluated
Python Patterns matchor is not evaluated


<a id="T08.02.7"></a>
## 08.02.7 · Its editions as data

`Python Patterns` covers the versions of Python that have a `match` statement, 3.10 to 3.15, each found by its
published name, every one with all eight of its kinds
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md));
Python 3.9 has none, so the dialect does not cover it:

In [8]:
dialect = store.Instances.Expressions.Python.Patterns
editions = Standards.standards(dialect)
print(", ".join(editions))
print("Python 3.10:", ", ".join(kind.name.rsplit(".", 1)[1] for kind in Standards.kinds(Standards.find(dialect, "Python 3.10"))))
print("Python 3.9 covered:", Standards.find(dialect, "Python 3.9") is not None)

Python 3.10, Python 3.11, Python 3.12, Python 3.13, Python 3.14, Python 3.15
Python 3.10: MatchValue, MatchSingleton, MatchSequence, MatchMapping, MatchClass, MatchStar, MatchAs, MatchOr
Python 3.9 covered: False


## What you learned

- **A pattern is a grammar of its own**, `Python Patterns`: eight kinds, from a capture and the wildcard to sequence,
  mapping, class, or and as patterns, which nest.
- **It holds Python3's terms**, a value's, a class's and a key's, and Python reads those as expressions; a pattern's own
  text is the `match` statement's, mbse-programs'.
- **It binds names, but is no binder**: `bound_names` gives what it binds, for the `case` that holds it, and
  `Symbolics.free` what it reads.
- **Validation speaks Python's rules**, `_` captures nothing among them; Python3's rules fold none of its kinds: each stays in the residual, named by its dialect.
- **Its editions are data**: Python 3.10 to 3.15.

Next, `Python RegExp`: the regular expressions of Python's `re`.

<!-- nav -->
[← 8.01 · Python Types](08_01_Python_Types.ipynb) · [Home](../../README.md) · [8.03 · Python RegExp →](08_03_Python_RegExp.ipynb)